# DATT - Colab T4 / CUDA 12
Run cells **1 -> 2 -> 2b (git pull) -> 3 -> 4 -> 5 -> 6** on a fresh GPU runtime. Do not run installation cells from the old notebook.
This notebook prepares the environment and starts DATT. Select your video in the dashboard.

- ORT GPU 1.26.0 was verified with Torch cu128 / cuDNN 9.
- GPU models live only in the DATT process; no duplicate FaceAnalysis/EasyOCR in the notebook kernel.
- Cell 2b pulls the latest selected branch with --ff-only. Stop DATT before updating. No automatic reset or service stop. /content is lost when the runtime is deleted.
- The GPU check stops if the CUDA major version changes.


In [ ]:
# 1. Configuration and GPU check in a fresh subprocess
import os, sys, subprocess, pathlib, socket, time, json
ROOT = pathlib.Path('/content/DATT')
REPO = 'https://github.com/kraken2811/DATT.git'
BRANCH = 'main'
ORT_VERSION = '1.26.0'
def run_logged(args, check=True, **kwargs):
    # Pipe child output through Python so Colab displays stderr as well as stdout.
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if check and result:
        raise subprocess.CalledProcessError(result, args)
    return result
def run_python(source):
    run_logged([sys.executable, '-u', '-c', source], check=True, cwd=ROOT if ROOT.exists() else None)
run_python("""
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
assert torch.version.cuda.startswith('12.'), 'This notebook targets CUDA 12'
print('GPU:', torch.cuda.get_device_name(0), 'cuDNN:', torch.backends.cudnn.version())
""")


In [ ]:
# 2. Clone only if missing; preserve an existing checkout
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(ROOT)], check=True)
assert (ROOT / 'src/main.py').is_file(), 'DATT project not found'
os.chdir(ROOT)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
# 2b. Update the selected branch from GitHub before installing or starting DATT
# Stop DATT with the optional stop cell before updating a running checkout.
for port in (8000, 8501):
    with socket.socket() as sock:
        assert sock.connect_ex(('127.0.0.1', port)) != 0, (
            f'Port {port} is occupied. Stop DATT before pulling code, then rerun this cell.'
        )
current_branch = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=ROOT, text=True
).strip()
assert current_branch == BRANCH, f'Expected branch {BRANCH}, found {current_branch!r}'
# Fast-forward only: never reset, auto-stash, discard edits or create a merge commit.
# If local edits conflict, git stops and preserves them for review.
run_logged(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT)
run_logged(['git', 'log', '-1', '--oneline'], cwd=ROOT)

# Compatibility fix for the currently published main branch (no git push needed).
app_path = ROOT / 'app.py'
app_source = app_path.read_text()
old_count = '            # People Occupancy\n            people_in_view, visible_person_ids = counter.update_and_get_visible_ids('
fixed_count = '            # Respect full-view / zone mode.\n            counter.polygon = np.asarray(config.ZONE_POLYGON, dtype=np.int32) if is_zone_on and config.ZONE_POLYGON is not None else None\n            people_in_view, visible_person_ids = counter.update_and_get_visible_ids('
if old_count in app_source:
    assert app_source.count(old_count) == 1 and 'import numpy as np' in app_source
    app_source = app_source.replace(old_count, fixed_count)
    compile(app_source, str(app_path), 'exec')
    app_path.write_text(app_source)
    print('Applied full-view people counter compatibility fix.')


In [ ]:
if 'datt_process' in globals() and datt_process.poll() is None:
    print('DATT is already running; skip this step and continue to dashboard.')
else:
    # 3. Install dependencies once; install the verified GPU ORT wheel LAST
    # Do not replace packages while a service is using these ports.
    for port in (8000, 8501):
        with socket.socket() as sock:
            assert sock.connect_ex(('127.0.0.1', port)) != 0, f'Port {port} is occupied; use a fresh runtime or stop its service first'
    assert 'onnxruntime' not in sys.modules and 'torch' not in sys.modules, 'Restart kernel before installation; do not reload native modules'
    # Preserve the Torch and torchvision versions supplied by Colab.
    import importlib.metadata as metadata
    constraints = ROOT / '.colab-constraints.txt'
    constraints.write_text('\n'.join([
        'torch==' + metadata.version('torch'),
        'torchvision==' + metadata.version('torchvision'),
        'onnxruntime-gpu==' + ORT_VERSION,
    ]) + '\n')
    # NumPy 1.x has no Python 3.13 wheel; use the appropriate Python branch.
    req = (ROOT / 'requirements-colab.txt').read_text()
    req = '\n'.join(
        'numpy<2.0.0; python_version < "3.13"\nnumpy>=2,<3; python_version >= "3.13"'
        if line.strip() == 'numpy<2.0.0' else line
        for line in req.splitlines()
    ) + '\n'
    install_req = ROOT / '.colab-install.txt'
    install_req.write_text(req)
    run_logged([sys.executable, '-m', 'pip', 'install', '-c', str(constraints), '-r', str(install_req),
        'fastapi', 'uvicorn[standard]', 'python-multipart', 'sqlalchemy', 'aiosqlite',
        'psycopg[binary]', 'pgvector', 'alembic', 'streamlink', 'huggingface_hub', 'yt-dlp[default]', 'deno'], check=True)
    # InsightFace can install CPU ORT in the same namespace; reinstall GPU ORT last.
    run_logged([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=True)
    run_logged([sys.executable, '-m', 'pip', 'install', '--no-deps', 'onnxruntime-gpu==' + ORT_VERSION], check=True)
    run_python("import torch, onnxruntime as ort; print('ORT:', ort.__version__, ort.get_available_providers()); assert 'CUDAExecutionProvider' in ort.get_available_providers()")


In [ ]:
# 4. Download missing models without holding GPU models in the kernel
run_python("""
from pathlib import Path
import shutil
from ultralytics.utils.downloads import attempt_download_asset
from huggingface_hub import hf_hub_download
root = Path('models')
root.mkdir(exist_ok=True)
yolo = root / 'yolo11s.pt'
if not yolo.exists() or yolo.stat().st_size == 0:
    src = Path(attempt_download_asset('yolo11s.pt'))
    if src.resolve() != yolo.resolve(): shutil.copy2(src, yolo)
plate = root / 'yolov8n-license-plate.pt'
if not plate.exists() or plate.stat().st_size == 0:
    shutil.copy2(hf_hub_download(repo_id='joker5914/yolov8n-license-plate', filename='best.pt'), plate)
for p in (yolo, plate): print(p, p.stat().st_size, 'bytes')
""")


In [ ]:
if 'datt_process' in globals() and datt_process.poll() is None:
    print('DATT is already running; skip this step and continue to dashboard.')
else:
    # 5. Initialize and verify sessions inside the actual DATT process
    for port in (8000, 8501):
        with socket.socket() as sock:
            assert sock.connect_ex(('127.0.0.1', port)) != 0, f'Port {port} is occupied; do not start a duplicate server'
    startup = """
    import sys, runpy, torch, onnxruntime as ort
    from src.face.face_embedder import FaceEmbedder
    assert torch.cuda.is_available()
    print('DATT ORT:', ort.__version__, 'available:', ort.get_available_providers(), flush=True)
    fe = FaceEmbedder.get_instance()
    assert fe.initialize(), fe._init_error
    for name, model in fe._app.models.items():
        providers = model.session.get_providers()
        print('DATT SESSION', name, providers, model.session.get_provider_options(), flush=True)
        if name in ('detection', 'recognition'):
            assert providers[0] == 'CUDAExecutionProvider', name + ' fell back to CPU'
    assert fe._det_model is not None and fe._rec_model is not None
    sys.argv = ['src/main.py', '--port', '8501', '--ai-port', '8000']
    runpy.run_path('src/main.py', run_name='__main__')
    """
    LOG = ROOT / 'datt-colab.log'
    with LOG.open('w') as output:
        datt_process = subprocess.Popen([sys.executable, '-u', '-c', __import__('textwrap').dedent(startup)], cwd=ROOT,
            stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
    print('DATT PID:', datt_process.pid, 'Log:', LOG)


In [ ]:
# 6. Wait for both services, then open the dashboard
import requests
from google.colab.output import serve_kernel_port_as_window
deadline = time.monotonic() + 300
ready = False
while time.monotonic() < deadline:
    if datt_process.poll() is not None:
        break
    try:
        ready = all(requests.get(url, timeout=2).ok for url in
            ['http://127.0.0.1:8000/telemetry', 'http://127.0.0.1:8501/'])
        if ready: break
    except requests.RequestException:
        pass
    time.sleep(2)
print(LOG.read_text(errors='replace')[-14000:])
assert ready, 'Not ready; inspect the log. Do not launch another server.'
serve_kernel_port_as_window(8501)


## Optional tools
These two cells do nothing during Run all unless their switches are enabled.
Do not install or upgrade ORT after cell 3. To restart DATT, stop it with the last cell, then run **5 -> 6** only.


In [ ]:
# 7. Inspect status and logs on demand
SHOW_DIAGNOSTICS = False
if SHOW_DIAGNOSTICS:
    print(LOG.read_text(errors='replace')[-14000:])
    for path in ('telemetry', 'api/source_status'):
        response = requests.get('http://127.0.0.1:8000/' + path, timeout=10)
        print(path, response.status_code, response.text[:4000])


In [ ]:
# 8. Stop only the process created by this notebook; no broad pkill
STOP_DATT = False
if STOP_DATT:
    import signal
    if 'datt_process' in globals() and datt_process.poll() is None:
        datt_process.send_signal(signal.SIGINT)
        try:
            datt_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            raise RuntimeError('DATT has not exited; inspect the log before continuing')
    print('Notebook DATT process stopped')
